# RAG Module · Notebook 4 · RAG Evaluation
Measure what your RAG system gets right — and where it fails.

---
> **Setup:** Run the first two cells before anything else.
> API key is loaded automatically from the `.env` file in the parent folder.

In [1]:
%pip install -q google-genai python-dotenv chromadb

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, math, time, base64, getpass, re, urllib.request
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
api_key = os.environ.get('GEMINI_API_KEY')
if not api_key:
    api_key = getpass.getpass('Paste your Gemini API key: ')

client = genai.Client(api_key=api_key)
MODEL = 'gemma-4-31b-it'
EMBED_MODEL = 'gemini-embedding-001'
DEFAULT_MAX = 2048

def cfg(**kwargs):
    kwargs.setdefault('max_output_tokens', DEFAULT_MAX)
    kwargs.setdefault('temperature', 0.7)
    return types.GenerateContentConfig(**kwargs)

def get_text(response) -> str:
    if response.text:
        return response.text.strip()
    parts = []
    for cand in (response.candidates or []):
        if cand.content:
            for part in cand.content.parts:
                if not getattr(part, 'thought', False) and part.text:
                    parts.append(part.text)
    return ''.join(parts).strip()

def _call_with_retry(fn, *args, max_retries=5, **kwargs):
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            msg = str(e)
            if '429' in msg or 'RESOURCE_EXHAUSTED' in msg:
                m = re.search(r'retry[^0-9]*([0-9]+)s', msg, re.I)
                wait = int(m.group(1)) + 5 if m else 35
                print(f'  ⏳ Rate-limited — waiting {wait}s (attempt {attempt+1}/{max_retries})')
                time.sleep(wait)
            else:
                raise
    raise RuntimeError('Max retries exceeded')

_raw_gen    = client.models.generate_content
_raw_stream = client.models.generate_content_stream
_raw_embed  = client.models.embed_content
_raw_count  = client.models.count_tokens
client.models.generate_content        = lambda *a,**kw: _call_with_retry(_raw_gen,    *a,**kw)
client.models.generate_content_stream = lambda *a,**kw: _call_with_retry(_raw_stream, *a,**kw)
client.models.embed_content           = lambda *a,**kw: _call_with_retry(_raw_embed,  *a,**kw)
client.models.count_tokens            = lambda *a,**kw: _call_with_retry(_raw_count,  *a,**kw)

print('✅ Setup complete. Model:', MODEL, '| Embed:', EMBED_MODEL, '| Retry-on-429: enabled')

✅ Setup complete. Model: gemma-4-31b-it | Embed: gemini-embedding-001 | Retry-on-429: enabled


### 🔌 API Key Test

In [3]:
try:
    _r = client.models.generate_content(
        model=MODEL,
        contents="Reply with exactly the words: Hello LLM course",
        config=cfg(temperature=0.0)
    )
    print("✅ API key working!")
    print("Model says:", get_text(_r))
except Exception as e:
    print("❌ API key error:", e)

✅ API key working!
Model says: Hello LLM course


### Rebuilding the RAG pipeline from Notebook 3

This notebook is self-contained. We rebuild the same corpus and pipeline here.

In [4]:
import chromadb

# ----- Embed helper -----
def embed(text: str) -> list:
    result = client.models.embed_content(model=EMBED_MODEL, contents=text)
    return result.embeddings[0].values

# ----- Chunking -----
def chunk_overlap(text: str, chunk_size: int = 300, overlap: int = 60) -> list:
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end].strip())
        start += chunk_size - overlap
    return [c for c in chunks if len(c) > 20]

# ----- Same corpus as Notebook 3 -----
CORPUS = {
    "python_history": """Python is a high-level, general-purpose programming language created by Guido van Rossum.
Development began in the late 1980s and the first version was released in 1991.
The name Python was inspired by the British comedy group Monty Python, not the snake.
Guido van Rossum served as Python's Benevolent Dictator For Life (BDFL) until 2018.
Python 2 was released in 2000 and Python 3 in 2008. Python 2 reached end-of-life in January 2020.""",

    "python_philosophy": """Python's design philosophy is captured in the Zen of Python (PEP 20).
Key principles include: Beautiful is better than ugly. Explicit is better than implicit.
Simple is better than complex. Readability counts. There should be one obvious way to do it.
Python emphasizes code readability and developer productivity over raw execution speed.
The language uses significant whitespace (indentation) to delimit code blocks.""",

    "python_data_types": """Python has several built-in data types: integers, floats, strings, booleans, and None.
Collection types include lists (mutable, ordered), tuples (immutable, ordered),
sets (unordered, unique elements), and dictionaries (key-value pairs).
Python uses dynamic typing — variables do not need type declarations.
Type hints were introduced in Python 3.5 via PEP 484 for optional static analysis.""",

    "python_web": """Django is a high-level Python web framework that encourages rapid development.
Flask is a lightweight micro-framework for building web applications in Python.
FastAPI is a modern, fast web framework for building APIs with Python type hints.
Python web frameworks follow WSGI or ASGI standards for server communication.
Django includes an ORM, admin interface, authentication, and many built-in features.""",

    "python_data_science": """NumPy provides efficient array operations and mathematical functions for Python.
Pandas offers data structures (DataFrame, Series) for data manipulation and analysis.
Matplotlib and Seaborn are popular Python libraries for data visualization.
Scikit-learn provides machine learning algorithms for classification, regression, and clustering.
Jupyter notebooks are widely used in data science for interactive Python code and visualization.""",

    "python_ml": """TensorFlow is an open-source machine learning framework developed by Google.
PyTorch is a machine learning framework developed by Meta, popular in research.
Both TensorFlow and PyTorch support GPU acceleration via CUDA for faster training.
Hugging Face Transformers provides pre-trained models for NLP tasks in Python.
Python is the dominant language for machine learning and artificial intelligence.""",

    "python_packaging": """pip is the standard package manager for Python, installing packages from PyPI.
PyPI (Python Package Index) hosts hundreds of thousands of third-party packages.
Virtual environments (venv, conda) isolate project dependencies to avoid conflicts.
pyproject.toml is the modern standard for Python project configuration (PEP 621).
Poetry and uv are modern alternatives to pip for dependency management.""",

    "python_async": """Python's asyncio library enables asynchronous programming with async/await syntax.
The Global Interpreter Lock (GIL) limits true CPU parallelism in CPython.
asyncio is ideal for I/O-bound tasks (network requests, file operations).
For CPU-bound tasks, the multiprocessing module bypasses the GIL using separate processes.
Python 3.12 introduced per-interpreter GIL, enabling true thread parallelism in subinterpreters.""",

    "python_testing": """pytest is the most popular testing framework for Python applications.
unittest is Python's built-in testing framework, part of the standard library.
Test coverage is measured with the coverage.py tool, often integrated with pytest.
Mocking in Python tests uses unittest.mock or the pytest-mock plugin.
TDD (Test-Driven Development) is widely practised in the Python community.""",

    "python_performance": """Python is slower than compiled languages like C, C++, or Rust due to interpretation.
List comprehensions and generator expressions are faster than equivalent for-loops.
The bisect module provides binary search for sorted lists in O(log n) time.
CPython is the reference implementation; PyPy offers a JIT compiler for significant speedups.
Profiling tools like cProfile and line_profiler help identify bottlenecks.""",

    "python_typing": """Python 3.5 introduced type hints with PEP 484. Type hints are optional annotations.
mypy is the most popular static type checker for Python.
The typing module provides generic types like List, Dict, Optional, and Union.
Python 3.10 added the union operator syntax: int | str instead of Union[int, str].
Dataclasses (Python 3.7+) generate __init__, __repr__, and other methods from type annotations.""",

    "python_standard_library": """Python's standard library is described as 'batteries included' — it covers a huge range.
Key modules: os (file system), sys (interpreter), json (JSON parsing), re (regex),
datetime (dates/times), pathlib (paths), collections (deque, Counter, defaultdict),
itertools (combinatorial iterators), functools (higher-order functions).
The standard library is always available — no pip install required.""",
}

# ----- Build ChromaDB collection -----
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="python_knowledge_eval")

chunk_id = 0
all_chunks_index = {}  # chunk_id → {text, source} for Precision@K labels

for doc_name, text in CORPUS.items():
    for chunk_text in chunk_overlap(text):
        cid = f"{doc_name}_{chunk_id}"
        vec = embed(chunk_text)
        collection.add(ids=[cid], documents=[chunk_text], embeddings=[vec], metadatas=[{"source": doc_name}])
        all_chunks_index[cid] = {"text": chunk_text, "source": doc_name}
        chunk_id += 1

print(f"✅ Indexed {collection.count()} chunks from {len(CORPUS)} documents.")

# ----- RAG pipeline (same as Notebook 3) -----
def retrieve(question: str, collection, top_k: int = 3) -> list:
    q_vec = embed(question)
    results = collection.query(query_embeddings=[q_vec], n_results=top_k,
                                include=['documents', 'distances', 'metadatas'])
    chunks = []
    for cid, doc, dist, meta in zip(results['ids'][0], results['documents'][0],
                                     results['distances'][0], results['metadatas'][0]):
        chunks.append({'id': cid, 'text': doc, 'distance': dist, 'source': meta.get('source', '')})
    return chunks

def rag_pipeline(question: str, collection, top_k: int = 3) -> dict:
    chunks = retrieve(question, collection, top_k=top_k)
    context = "\n\n".join(f"[Source {i+1}: {c['source']}]\n{c['text']}" for i, c in enumerate(chunks))
    prompt = f"""Answer the question using ONLY the context provided below.
If the answer is not in the context, respond with exactly:
"I don't have that information in my knowledge base."

Context:
{context}

Question: {question}

Answer:"""
    r = client.models.generate_content(model=MODEL, contents=prompt, config=cfg(temperature=0.0))
    return {'answer': get_text(r), 'chunks': chunks}

print("✅ RAG pipeline ready.")

✅ Indexed 24 chunks from 12 documents.
✅ RAG pipeline ready.


---
## 1. Why RAG Evaluation Matters — Two Things Can Go Wrong
Retrieval and generation are independent steps — each can fail independently.

**Developer analogy:** A broken phone line vs. a confused translator — you need to diagnose which one failed before you can fix it.

### The two failure modes

1. **Retrieval failure**: the right chunk exists in the DB but wasn't returned (wrong chunks retrieved)
2. **Generation failure**: the right chunks were retrieved but the model ignored them or hallucinated beyond them

```
User question
     ↓
 [RETRIEVAL] ← can fail here: wrong chunks returned
     ↓
 [GENERATION] ← can fail here: model ignores context or hallucinates
     ↓
 Answer
```

### Diagnosing failures

| What failed | Symptom | Fix |
|---|---|---|
| Retrieval | Answer is "I don't know" but the fact is in the DB | Improve chunking, increase top_k, use better embedding model |
| Generation | Answer is confident but wrong despite correct chunks | Lower temperature, strengthen grounding instruction, verify with judge |
| Both | Consistently wrong answers | Systematic evaluation (this notebook) |

### Metrics overview

| Metric | What it measures | Type |
|---|---|---|
| Precision@K | Fraction of retrieved chunks that are relevant | Retrieval |
| Recall@K | Fraction of relevant chunks that were retrieved | Retrieval |
| Faithfulness | Does the answer only use facts from the context? | Generation |
| Answer relevance | Does the answer actually address the question? | Generation |

---
## 2. Faithfulness — Did the Model Stick to the Evidence?
A faithful answer contains only claims that are supported by the retrieved context.

**Developer analogy:** A faithful answer is like citing your sources — every claim can be traced back to a retrieved chunk. An unfaithful answer invents details not in the context.

**Pattern:** generate answer → ask a judge LLM to audit each claim → SUPPORTED or NOT_SUPPORTED

| Faithfulness | Meaning | Action |
|---|---|---|
| FAITHFUL | Every claim is in the retrieved context | Answer is trustworthy |
| UNFAITHFUL | Model added facts not in the context | Lower temperature, strengthen grounding prompt |

In [5]:
def check_faithfulness(answer: str, context: str) -> dict:
    """
    Use an LLM judge to check whether every claim in the answer
    is supported by the provided context.
    Returns a dict with 'verdict' (FAITHFUL/UNFAITHFUL) and 'analysis'.
    """
    judge_prompt = f"""You are a factual accuracy auditor.

Given a context and an answer, check whether every factual claim in the answer
is directly supported by the context. 

For each claim in the answer:
- Label it SUPPORTED if it appears in the context (even if paraphrased)
- Label it NOT_SUPPORTED if it goes beyond what the context says

Then give an overall verdict: FAITHFUL if all claims are supported, UNFAITHFUL if any are not.

Context:
{context}

Answer to audit:
{answer}

Respond in this exact format:
CLAIMS:
- [claim]: SUPPORTED/NOT_SUPPORTED

VERDICT: FAITHFUL/UNFAITHFUL"""

    r = client.models.generate_content(
        model=MODEL,
        contents=judge_prompt,
        config=cfg(temperature=0.0)
    )
    raw = get_text(r)
    verdict = 'FAITHFUL' if 'VERDICT: FAITHFUL' in raw and 'UNFAITHFUL' not in raw.split('VERDICT:')[1] else 'UNFAITHFUL'
    return {'verdict': verdict, 'analysis': raw}

In [6]:
# Demo 1: A faithful answer (model uses only retrieved context)
question_faithful = "What is pytest used for in Python?"
result = rag_pipeline(question_faithful, collection)

context = "\n\n".join(c['text'] for c in result['chunks'])
print(f"Question: {question_faithful}")
print(f"\nAnswer:\n{result['answer']}")
print(f"\nRetrieved context preview:\n{context[:300]}...")

faith = check_faithfulness(result['answer'], context)
print(f"\n🔍 Faithfulness verdict: {faith['verdict']}")
print(faith['analysis'])

Question: What is pytest used for in Python?

Answer:
pytest is the most popular testing framework for Python applications.

Retrieved context preview:
pytest is the most popular testing framework for Python applications.
unittest is Python's built-in testing framework, part of the standard library.
Test coverage is measured with the coverage.py tool, often integrated with pytest.
Mocking in Python tests uses unittest.mock or the pytest-mock plugin...



🔍 Faithfulness verdict: FAITHFUL
CLAIMS:
- pytest is the most popular testing framework for Python applications: SUPPORTED

VERDICT: FAITHFUL


In [7]:
# Demo 2: Force an unfaithful answer by giving minimal context for a complex question
# We manually provide a context that's slightly off-topic, causing the model to hallucinate
misleading_context = """Python supports object-oriented programming through classes.
The standard library includes many useful modules."""

unfaithful_question = "What is the exact release date of Python 3.12 and what were its main features?"

r_unfaithful = client.models.generate_content(
    model=MODEL,
    contents=f"""Answer the question using ONLY the context provided below.
Context:
{misleading_context}

Question: {unfaithful_question}
Answer:""",
    config=cfg(temperature=0.7)  # higher temperature to encourage hallucination
)
unfaithful_answer = get_text(r_unfaithful)

print(f"Question: {unfaithful_question}")
print(f"\nAnswer (potentially unfaithful):\n{unfaithful_answer}")

faith2 = check_faithfulness(unfaithful_answer, misleading_context)
print(f"\n🔍 Faithfulness verdict: {faith2['verdict']}")
print(faith2['analysis'])
print("\n✅ The judge catches claims that go beyond the provided context.")

Question: What is the exact release date of Python 3.12 and what were its main features?

Answer (potentially unfaithful):
The provided context does not contain information regarding the release date or main features of Python 3.12.



🔍 Faithfulness verdict: FAITHFUL
CLAIMS:
- The provided context does not contain information regarding the release date of Python 3.12: SUPPORTED
- The provided context does not contain information regarding the main features of Python 3.12: SUPPORTED

VERDICT: FAITHFUL

✅ The judge catches claims that go beyond the provided context.


---
## 3. Retrieval Precision@K — Are We Retrieving the Right Chunks?
Precision@K measures what fraction of the top-K retrieved chunks are actually relevant to the question.

**Developer analogy:** If you ask a librarian for 3 books on Python and they return 2 Python books + 1 gardening book, Precision@3 = 2/3 = 0.67.

**Formula:** `Precision@K = (number of relevant chunks in top-K) / K`

| Concept | Practice |
|---|---|
| Relevant chunk | A chunk whose source document contains the answer to the question |
| K | The number of chunks retrieved (top_k parameter) |
| High Precision@K | Most retrieved chunks are on-topic |
| Low Precision@K | Retrieved chunks are off-topic — retrieval needs fixing |

In [8]:
def precision_at_k(retrieved_ids: list, relevant_sources: list, k: int = 3) -> float:
    """
    Compute Precision@K.
    retrieved_ids: list of chunk IDs returned by retrieval (ordered by relevance)
    relevant_sources: list of source document names that are relevant to the question
    k: how many top results to evaluate
    """
    top_k_ids = retrieved_ids[:k]
    # A chunk is relevant if its source document is in the relevant_sources list
    relevant_count = sum(
        1 for cid in top_k_ids
        if any(src in cid for src in relevant_sources)
    )
    return relevant_count / k if k > 0 else 0.0

In [9]:
# Ground truth: for each question, which source documents contain the answer?
TEST_SET = [
    {
        "question": "Who created Python and when was it first released?",
        "relevant_sources": ["python_history"],  # answer is in python_history doc
    },
    {
        "question": "What frameworks are available for Python web development?",
        "relevant_sources": ["python_web"],
    },
    {
        "question": "How does Python handle concurrency and what is the GIL?",
        "relevant_sources": ["python_async"],
    },
    {
        "question": "What tools does Python offer for testing code?",
        "relevant_sources": ["python_testing"],
    },
]

print(f"Test set: {len(TEST_SET)} questions with hand-labeled relevant sources\n")

results_precision = []
for test in TEST_SET:
    chunks = retrieve(test['question'], collection, top_k=3)
    retrieved_ids = [c['id'] for c in chunks]
    p_at_3 = precision_at_k(retrieved_ids, test['relevant_sources'], k=3)
    results_precision.append({
        'question': test['question'],
        'retrieved_sources': [c['source'] for c in chunks],
        'relevant_sources': test['relevant_sources'],
        'precision_at_3': p_at_3,
    })
    print(f"Q: {test['question'][:60]}...")
    print(f"   Retrieved: {[c['source'] for c in chunks]}")
    print(f"   Relevant:  {test['relevant_sources']}")
    print(f"   Precision@3: {p_at_3:.2f}\n")

avg_precision = sum(r['precision_at_3'] for r in results_precision) / len(results_precision)
print(f"Average Precision@3: {avg_precision:.2f}")

Test set: 4 questions with hand-labeled relevant sources



Q: Who created Python and when was it first released?...
   Retrieved: ['python_history', 'python_history', 'python_philosophy']
   Relevant:  ['python_history']
   Precision@3: 0.67



Q: What frameworks are available for Python web development?...
   Retrieved: ['python_web', 'python_web', 'python_testing']
   Relevant:  ['python_web']
   Precision@3: 0.67



Q: How does Python handle concurrency and what is the GIL?...
   Retrieved: ['python_async', 'python_async', 'python_performance']
   Relevant:  ['python_async']
   Precision@3: 0.67



Q: What tools does Python offer for testing code?...
   Retrieved: ['python_testing', 'python_testing', 'python_standard_library']
   Relevant:  ['python_testing']
   Precision@3: 0.67

Average Precision@3: 0.67


In [10]:
# ✏️ Add your own test case to measure retrieval precision
my_question = "What is NumPy used for in Python?"
my_relevant_sources = ["python_data_science"]  # which source doc contains the answer?

my_chunks = retrieve(my_question, collection, top_k=3)
my_ids = [c['id'] for c in my_chunks]
my_p = precision_at_k(my_ids, my_relevant_sources, k=3)

print(f"Question: {my_question}")
print(f"Retrieved: {[c['source'] for c in my_chunks]}")
print(f"Precision@3: {my_p:.2f}")

Question: What is NumPy used for in Python?
Retrieved: ['python_data_science', 'python_data_science', 'python_ml']
Precision@3: 0.67


---
## 4. End-to-End Evaluation — Putting It All Together
Combine retrieval precision and answer faithfulness into a single pass/fail evaluation for each test case.

This gives you a complete picture of where your RAG system succeeds and where it needs improvement.

| Failure | Precision@K | Faithful | Fix |
|---|---|---|---|
| Retrieval miss | Low | Any | Smaller chunks, larger top_k, rephrase query |
| Hallucination | Any | No | temperature=0.0, stronger grounding prompt |
| Both fail | Low | No | Fix retrieval first — generation can't work without the right context |
| All pass | High | Yes | Consider harder test cases or increasing corpus size |

In [11]:
def evaluate_rag(test_cases: list, collection, top_k: int = 3) -> list:
    """
    Run the full RAG pipeline on each test case and score:
    - Retrieval Precision@K
    - Answer Faithfulness (LLM-as-judge)
    Returns a list of result dicts.
    """
    results = []
    for i, test in enumerate(test_cases, 1):
        print(f"Evaluating {i}/{len(test_cases)}: {test['question'][:50]}...")

        # RAG pipeline
        rag_result = rag_pipeline(test['question'], collection, top_k=top_k)
        answer = rag_result['answer']
        chunks = rag_result['chunks']

        # Retrieval precision
        retrieved_ids = [c['id'] for c in chunks]
        p_at_k = precision_at_k(retrieved_ids, test['relevant_sources'], k=top_k)

        # Faithfulness
        context = "\n\n".join(c['text'] for c in chunks)
        faith_result = check_faithfulness(answer, context)
        is_faithful = faith_result['verdict'] == 'FAITHFUL'

        # Pass = both precision > 0.5 AND faithful
        passed = p_at_k >= 0.5 and is_faithful

        results.append({
            'question': test['question'],
            'answer': answer,
            'precision_at_k': p_at_k,
            'faithful': is_faithful,
            'passed': passed,
        })

    return results

In [12]:
print("Running end-to-end evaluation...\n")
eval_results = evaluate_rag(TEST_SET, collection)

print("\n" + "=" * 90)
print(f"{'Question':<45} {'P@3':>5} {'Faithful':>10} {'Pass':>6}")
print("=" * 90)

passed_count = 0
for r in eval_results:
    q_short = r['question'][:43] + ".." if len(r['question']) > 45 else r['question']
    faithful_str = "Yes" if r['faithful'] else "No"
    pass_str = "✅" if r['passed'] else "❌"
    if r['passed']:
        passed_count += 1
    print(f"{q_short:<45} {r['precision_at_k']:>5.2f} {faithful_str:>10} {pass_str:>6}")

print("=" * 90)
print(f"\nResult: {passed_count}/{len(eval_results)} test cases passed")
print(f"Average Precision@3: {sum(r['precision_at_k'] for r in eval_results)/len(eval_results):.2f}")

Running end-to-end evaluation...

Evaluating 1/4: Who created Python and when was it first released?...


Evaluating 2/4: What frameworks are available for Python web devel...


Evaluating 3/4: How does Python handle concurrency and what is the...


Evaluating 4/4: What tools does Python offer for testing code?...



Question                                        P@3   Faithful   Pass
Who created Python and when was it first re..  0.67        Yes      ✅
What frameworks are available for Python we..  0.67         No      ❌
How does Python handle concurrency and what..  0.67        Yes      ✅
What tools does Python offer for testing co..  0.67        Yes      ✅

Result: 3/4 test cases passed
Average Precision@3: 0.67


In [13]:
# Show improvement suggestions for any failing test cases
failing = [r for r in eval_results if not r['passed']]
if failing:
    print("Failing test cases and suggested fixes:\n")
    for r in failing:
        print(f"Q: {r['question'][:70]}")
        if r['precision_at_k'] < 0.5:
            print(f"  ❌ Retrieval issue (P@3={r['precision_at_k']:.2f})")
            print(f"     → Try: increase top_k, use smaller chunk_size, or add synonyms to the query")
        if not r['faithful']:
            print(f"  ❌ Faithfulness issue")
            print(f"     → Try: lower temperature (use 0.0), strengthen grounding prompt, or increase top_k")
        print()
else:
    print("✅ All test cases passed!")

Failing test cases and suggested fixes:

Q: What frameworks are available for Python web development?
  ❌ Faithfulness issue
     → Try: lower temperature (use 0.0), strengthen grounding prompt, or increase top_k



---
### RAG Module — Complete Summary

You've now built and evaluated a complete RAG system from scratch.

| Notebook | What you learned |
|---|---|
| 1 · Why RAG? | Hallucination, context limits, chunking, the case for RAG |
| 2 · Embeddings | Text → vectors, cosine similarity, semantic search |
| 3 · RAG Pipeline | ChromaDB, indexing, retrieval, augmented generation |
| 4 · Evaluation | Precision@K, faithfulness judging, end-to-end scoring |

**The RAG loop:**
```
Documents → Chunk → Embed → ChromaDB
                                ↓ (at query time)
Question → Embed → Retrieve top-K → Augment prompt → LLM → Grounded answer
                                                              ↓
                                              Evaluate: Precision@K + Faithfulness
```

**What's next:**
- Use `chromadb.PersistentClient(path="./db")` to persist your index across sessions
- Try larger corpora (PDFs, web pages) with a PDF parsing library like `pypdf`
- Experiment with re-ranking retrieved chunks before passing them to the LLM
- Explore hybrid search (keyword + semantic) for better retrieval on technical queries